# Mapping Canonical JSON to JSON-LD

This notebook demonstrates:
- direct mapper usage (`to_jsonld`)
- CLI mapping command
- comparison with golden mapping outputs


In [ ]:
from pathlib import Path
import json
import re
import sys
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    # If notebook is opened from inside notebooks/, move to repo root.
    ROOT = ROOT.parent

assert (ROOT / 'src').exists(), f'Repo root not found from {Path.cwd()}'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('ROOT:', ROOT)

from battinfo.transform.json_to_jsonld import to_jsonld

In [ ]:
input_doc = json.loads((ROOT / 'src' / 'battinfo' / 'data' / 'examples' / 'cells' / 'A123_20AH.curated.json').read_text(encoding='utf-8'))
map_domain = to_jsonld(input_doc, target='domain-battery')
map_bp = to_jsonld(input_doc, target='batterypass')
print('domain keys:', sorted(map_domain.keys()))
print('batterypass @type:', map_bp.get('@type'))
print('batterypass version:', map_bp.get('batterypass:version'))


In [ ]:
from battinfo.cli import app
from typer.testing import CliRunner

runner = CliRunner()
for target in ('domain-battery', 'batterypass'):
    out_file = ROOT / f'tmp_{target}.jsonld'
    result = runner.invoke(
        app,
        [
            'map',
            str(ROOT / 'src' / 'battinfo' / 'data' / 'examples' / 'cells' / 'A123_20AH.curated.json'),
            '--target',
            target,
            '--out',
            str(out_file),
        ],
    )
    print(target, 'exit:', result.exit_code)
    print(result.stdout.strip())


In [ ]:
golden_domain = json.loads((ROOT / 'tests' / 'golden' / 'domain_battery_a123_20ah.jsonld').read_text(encoding='utf-8'))
golden_bp = json.loads((ROOT / 'tests' / 'golden' / 'batterypass_a123_20ah.jsonld').read_text(encoding='utf-8'))

tmp_domain = json.loads((ROOT / 'tmp_domain-battery.jsonld').read_text(encoding='utf-8'))
tmp_bp = json.loads((ROOT / 'tmp_batterypass.jsonld').read_text(encoding='utf-8'))

print('domain equals golden:', tmp_domain == golden_domain)
print('batterypass equals golden:', tmp_bp == golden_bp)


In [ ]:
# Cleanup temp files
for f in [ROOT / 'tmp_domain-battery.jsonld', ROOT / 'tmp_batterypass.jsonld']:
    if f.exists():
        f.unlink()
print('done')


## Debug tip

If mappings diverge from golden files, inspect:
- `src/battinfo/transform/json_to_jsonld.py`
- `tests/golden/*.jsonld`
